# 06 多个中间件组合使用及执行顺序

> 单个中间件能力有限，**组合使用**才是生产级 Agent 的常态。
> 本文件讲解：
> 1. 中间件的两种执行机制（钩子型 / 包装型）；
> 2. **执行顺序规则**（`middleware=[...]` 列表顺序如何影响执行）；
> 3. 多个中间件组合的完整示例；
> 4. 顺序调换带来的行为差异（真实踩坑点）。

---

## 1. 两种执行机制

| 机制 | 钩子名 | 特点 |
| --- | --- | --- |
| **钩子型（hook）** | `before_agent` / `after_agent`、`before_model` / `after_model` | 在 Agent 图的节点**前后插入**逻辑，可读写 state、可 `jump_to` 跳转 |
| **包装型（wrap）** | `wrap_model_call` / `wrap_tool_call` | 把“真实调用”**包起来**，可修改请求、短路、重试、回退 |

两种机制都会在 `middleware=[...]` 列表中**按顺序组合**，规则如下。

---

## 2. 执行顺序核心规则

### 规则 1：`before_model` 按列表顺序执行

```python
middleware = [A, B, C]
# 执行顺序：A.before_model → B.before_model → C.before_model → 模型调用
```

### 规则 2：`after_model` 按列表逆序执行

```python
# 执行顺序：模型调用 → C.after_model → B.after_model → A.after_model
```

### 规则 3：`wrap_model_call` / `wrap_tool_call` —— 列表第一个 = 最外层（洋葱模型）

```python
# middleware = [A, B, C] 时，A 是最外层，C 最靠近真实调用
#
# 真实调用
#    ▲
#    │ C.wrap_model_call  （最内层）
#    │ B.wrap_model_call
#    │ A.wrap_model_call  （最外层，最先进入）
#    ▲
# 进入顺序：A → B → C → 真实调用 → C → B → A 返回
```

**一句话记忆**：`before_model` 正向、`after_model` 反向、`wrap_*` 洋葱套娃（第一个最外层）。

### 为什么这样设计？

- **before 正向**：先声明的中间件先做“前置检查/准备”，后面的接着做，最后才轮到模型；
- **after 反向**：后声明的中间件先处理“模型输出”，先声明的最后做“收尾”，保证与进入顺序对称；
- **wrap 洋葱**：越靠前的中间件越能“包裹”后面的所有逻辑（比如最外层做整体兜底回退）。

---

## 3. 组合使用示例 1：TraceMiddleware 亲眼观察执行顺序

下面自定义一个“记录日志”的教学中间件，通过打印日志把上面三条规则**可视化**。

In [ ]:
# ============================================================
# 环境准备（与前面文件一致）
# ============================================================
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    LLMToolSelectorMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    ToolRetryMiddleware,
)
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)

llm = init_chat_model(model="deepseek:deepseek-v4-flash")
print("环境准备完成")

In [ ]:
@tool
def get_weather(city: str) -> str:
    """查询指定城市当前的天气情况。"""
    return f"{city}：晴，26℃，微风"


@tool
def get_time(city: str) -> str:
    """查询指定城市当前的本地时间。"""
    return f"{city} 当前时间：14:30"


@tool
def calculator(expression: str) -> str:
    """计算数学表达式。"""
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:  # noqa: BLE001
        return f"计算失败：{e}"


ALL_TOOLS = [get_weather, get_time, calculator]
print("工具：", [t.name for t in ALL_TOOLS])

In [ ]:
# ============================================================
# 自定义一个“执行顺序可视化”中间件 TraceMiddleware
#
# 同时实现 4 个钩子，打印进入/离开日志，用来观察顺序规则。
# 注意 name 属性：允许同名类创建多个实例（用 label 区分），
# 否则 create_agent 会因“重复中间件实例”报错。
# ============================================================

class TraceMiddleware(AgentMiddleware):
    """打印各钩子执行顺序的教学中间件。"""

    def __init__(self, label: str):
        super().__init__()
        self.label = label

    @property
    def name(self) -> str:
        # 每个实例用 label 区分，避免 create_agent 的“重名检查”报错
        return f"TraceMiddleware[{self.label}]"

    # ---- 钩子型 ----
    def before_model(self, state, runtime):
        rprint(f"[bold cyan]▶ {self.label}.before_model[/]  在模型调用之前")
        return None  # 不修改状态

    async def abefore_model(self, state, runtime):
        return self.before_model(state, runtime)

    def after_model(self, state, runtime):
        rprint(f"[bold magenta]◀ {self.label}.after_model[/]  在模型调用之后")
        return None

    async def aafter_model(self, state, runtime):
        return self.after_model(state, runtime)

    # ---- 包装型 ----
    def wrap_model_call(self, request, handler):
        rprint(f"[bold yellow]◉ {self.label}.wrap_model_call[/]  进入（包裹真实模型调用）")
        result = handler(request)  # 调用内层
        rprint(f"[bold yellow]◉ {self.label}.wrap_model_call[/]  返回")
        return result

    async def awrap_model_call(self, request, handler):
        rprint(f"[bold yellow]◉ {self.label}.wrap_model_call[/]  进入（包裹真实模型调用）")
        result = await handler(request)
        rprint(f"[bold yellow]◉ {self.label}.wrap_model_call[/]  返回")
        return result


print("TraceMiddleware 定义完成")

In [ ]:
# ============================================================
# 用 [A, B, C] 三个 TraceMiddleware 组合，观察执行顺序
#
# 预期打印顺序：
#   before_model：A → B → C
#   wrap_model_call 进入：A → B → C
#   wrap_model_call 返回：C → B → A
#   after_model：C → B → A   （注意是逆序！）
# ============================================================
agent_trace = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[
        TraceMiddleware("A"),
        TraceMiddleware("B"),
        TraceMiddleware("C"),
    ],
)

resp = await agent_trace.ainvoke({
    "messages": [
        HumanMessage(content="查询北京天气")
    ]
})

print("\n=== 最终回答 ===")
resp["messages"][-1].pretty_print()

# 请对照上方日志，验证三条核心顺序规则是否与文档描述一致！

---

## 4. 组合设计建议（生产经验）

多个中间件组合时，建议按“**外层放全局策略，内层放局部能力**”的原则排列：

| 层级 | 推荐中间件 | 理由 |
| --- | --- | --- |
| 最外层 | `ModelFallbackMiddleware` / `ModelRetryMiddleware(on_failure="error")` | 全局兜底：任何内层失败都能被回退/重试接住 |
| 次外层 | `LLMToolSelectorMiddleware` | 先裁剪工具列表，让后续环节只面对精简工具 |
| 中层 | `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | 限流防失控（它们走钩子节点，顺序与 wrap 链独立） |
| 最内层 | `ToolRetryMiddleware` | 只针对单次工具调用做重试 |

> ⚠️ **关键坑**：`ModelRetryMiddleware` 默认 `on_failure="continue"` 会把“重试耗尽”**吞成一条错误消息返回**，
> 此时外层 `ModelFallbackMiddleware` **收不到异常、不会回退**。
> 要让“先重试、再回退”生效，必须把重试中间件设成 `on_failure="error"`（见示例 3）。

---

## 5. 组合示例 2：工具选择 + 调用限流 + 失败重试（生产组合）

一个“多功能助手”的完整中间件栈：
1. `LLMToolSelectorMiddleware`：从 5 个工具中挑最相关的（省 token）；
2. `ToolCallLimitMiddleware`：单次运行最多 4 次工具调用（防失控）；
3. `ModelCallLimitMiddleware`：单次运行最多 6 次模型调用（防失控）；
4. `ToolRetryMiddleware`：工具调用失败自动重试 2 次；
5. `ModelRetryMiddleware`：模型调用失败自动重试 1 次。

In [ ]:
combined_mw = [
    # ---- 外层：先裁剪工具列表 ----
    LLMToolSelectorMiddleware(
        max_tools=3,
        always_include=["get_weather"],
    ),
    # ---- 中层：限流防失控（钩子型） ----
    ToolCallLimitMiddleware(
        run_limit=4,
        exit_behavior="continue",   # 超限后阻断多余调用，模型可继续收尾
    ),
    ModelCallLimitMiddleware(
        run_limit=6,
        exit_behavior="end",        # 模型调用超限后温和结束
    ),
    # ---- 内层：失败重试 ----
    ToolRetryMiddleware(
        max_retries=2,
        initial_delay=0.2,
        backoff_factor=1.0,
    ),
    ModelRetryMiddleware(
        max_retries=1,
        initial_delay=0.2,
        backoff_factor=1.0,
    ),
]

agent_combined = create_agent(
    model=llm,
    tools=ALL_TOOLS,
    middleware=combined_mw,
)

resp_c = await agent_combined.ainvoke({
    "messages": [
        HumanMessage(content="分别查询北京和上海的天气，对比一下哪个更适合外出，最后算一下 25 的平方")
    ]
})

for msg in resp_c["messages"]:
    msg.pretty_print()

---

## 6. 示例 3：组合不当导致“回退失效”的真实踩坑点

以 **“主模型必然失败”** 的场景为例。回退要生效，必须让“重试耗尽后的异常”能被外层回退中间件捕获：

- **方案一（错误示范）**：`[ModelFallbackMiddleware, ModelRetryMiddleware]`（重试默认 `on_failure="continue"`）
  - 主模型失败 → 内层重试接管，重试耗尽后**把失败“吞”成一条错误消息返回**（不抛异常）
  - ❌ 外层回退中间件看到的是一次“成功”，**回退根本不会触发**，用户只会收到一条错误消息
- **方案二（正确示范）**：`[ModelFallbackMiddleware, ModelRetryMiddleware(on_failure="error")]`
  - 主模型失败 → 内层重试接管，重试耗尽后 `on_failure="error"` **重新抛出异常**
  - ✅ 外层回退中间件捕获异常 → 切换到备用模型 → 正常回答
- 补充：如果写成 `[ModelRetryMiddleware, ModelFallbackMiddleware]`（回退排在最内层），
  主模型一失败就会被内层回退**抢先**切换到备用模型，主模型连重试机会都没有 —— 顺序语义同样不同。

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult


class AlwaysFailChatModel(BaseChatModel):
    """任何调用都会抛异常的主模型，用来制造“模型失败”场景。"""

    @property
    def _llm_type(self) -> str:
        return "always-fail"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        raise RuntimeError("主模型服务不可用（模拟故障）")


# ---------- 方案一（错误示范）：回退在外层 + 重试在内层（默认 on_failure="continue" 吞异常） ----------
bad_order = [
    ModelFallbackMiddleware("deepseek:deepseek-v4-flash"),    # 外层：整体回退
    ModelRetryMiddleware(max_retries=1, initial_delay=0.1),   # 内层：重试（默认 continue）
]

agent_bad = create_agent(
    model=AlwaysFailChatModel(),
    middleware=bad_order,
)

resp_bad = await agent_bad.ainvoke({
    "messages": [HumanMessage(content="你好，请自我介绍")]
})

# 观察：Agent 并没有真正切换到备用模型，而是收到了“重试耗尽”的错误消息
rprint("[bold red]方案一最终回复（重试耗尽返回的错误消息，回退未生效）：[/]")
resp_bad["messages"][-1].pretty_print()

In [ ]:
# ---------- 方案二（正确示范）：回退在外层 + 重试 on_failure="error" ----------
good_order = [
    ModelFallbackMiddleware("deepseek:deepseek-v4-flash"),   # 外层：整体回退
    ModelRetryMiddleware(
        max_retries=1,
        initial_delay=0.1,
        on_failure="error",   # 关键：重试耗尽后重新抛异常，让外层回退能感知
    ),
]

agent_good = create_agent(
    model=AlwaysFailChatModel(),
    middleware=good_order,
)

resp_good = await agent_good.ainvoke({
    "messages": [HumanMessage(content="你好，请自我介绍")]
})

# 观察：这次主模型失败 → 重试 1 次 → 抛错 → 回退到 deepseek → 正常回答
rprint("[bold green]方案二最终回复（先重试、再回退，回退已生效）：[/]")
resp_good["messages"][-1].pretty_print()

---

## 7. 注意事项与易错点汇总

1. **顺序 = 列表顺序**：`middleware=[...]` 的顺序直接影响执行逻辑，先想清楚再排列。
2. **`before_model` 正向、`after_model` 反向**：需要“先进入的后收尾”的效果，就按直觉排列即可；
   需要精确控制时请用 TraceMiddleware 这类日志中间件验证。
3. **wrap 链第一个 = 最外层**：兜底类中间件（回退、全局重试）请放前面。
4. **`on_failure` 会吞异常**：`"continue"` 会把失败转成消息返回，外层无法感知；
   要与回退类中间件配合时，务必设成 `"error"`。
5. **不能有重名实例**：`create_agent` 会校验中间件名称唯一（`name` 属性默认是类名），
   同一种中间件要挂多个实例时，请重写 `name` 属性。
6. **钩子型与包装型并存**：钩子型是图上的独立节点，包装型是调用链上的包裹层，
   两者独立编排、互不冲突。
7. **测试建议**：组合逻辑复杂时，先只挂 TraceMiddleware 观察顺序，再加真实中间件，
   逐层叠加、逐步验证。

---

## 总结

| 知识点 | 结论 |
| --- | --- |
| 组合方式 | `create_agent(middleware=[mw1, mw2, ...])` 传入列表即可 |
| before_model | 按列表顺序执行 |
| after_model | 按列表逆序执行 |
| wrap_model_call / wrap_tool_call | 列表第一个 = 最外层（洋葱模型） |
| 兜底类中间件 | 放最外层 |
| 限流类中间件 | 放中层（防失控） |
| 重试 + 回退组合 | 回退在外、重试设 `on_failure="error"` 在内 |

> 本文件完。掌握顺序规则后，你就能像搭积木一样自由组合官方中间件，构建生产级 Agent。